# SQL Analysis with SQLite

Loads the CSV into an in-memory SQLite database and runs analytical queries.

In [1]:
import pandas as pd
import numpy as np
import sqlite3
import os, warnings, io
warnings.filterwarnings('ignore')

try:
    df = pd.read_csv('data/demand_clean.csv', parse_dates=['Date'])
except FileNotFoundError:
    from google.colab import files
    print("Upload demand_clean.csv")
    uploaded = files.upload()
    df = pd.read_csv(io.BytesIO(list(uploaded.values())[0]), parse_dates=['Date'])

# load into SQLite
conn = sqlite3.connect(":memory:")
df['Date'] = df['Date'].dt.strftime('%Y-%m-%d')
df.to_sql('demand', conn, index=False, if_exists='replace')
print("Loaded", len(df), "rows into SQLite")

Loaded 1048575 rows into SQLite


## Query 1 – Row Count

In [2]:
pd.read_sql('SELECT COUNT(*) AS total_rows FROM demand', conn)

,total_rows
0,1048575


## Query 2 – Total Demand per Warehouse

In [3]:
q = '''
SELECT Warehouse,
       SUM(Order_Demand)  AS total_demand,
       COUNT(*)           AS order_count,
       AVG(Order_Demand)  AS avg_demand
FROM demand
GROUP BY Warehouse
ORDER BY total_demand DESC
'''
pd.read_sql(q, conn)

,Warehouse,total_demand,order_count,avg_demand
0,Whse_J,3363200396,764447,4399.520694
1,Whse_S,1038024700,88200,11768.987528
2,Whse_C,585071404,42354,13813.840582
3,Whse_A,159036821,153574,1035.571262


## Query 3 – Demand by Product Category

In [4]:
q = '''
SELECT Product_Category,
       SUM(Order_Demand) AS total_demand,
       COUNT(*)          AS orders
FROM demand
GROUP BY Product_Category
ORDER BY total_demand DESC
'''
pd.read_sql(q, conn)

,Product_Category,total_demand,orders
0,Category_019,4251207605,481099
1,Category_006,405579330,35577
2,Category_005,199681320,101671
3,Category_007,128691531,82402
4,Category_028,49150112,31302
5,Category_033,42610000,1849
6,Category_030,40966555,12997
7,Category_021,4480660,52011
8,Category_032,4473048,9296
9,Category_009,3782141,19738


## Query 4 – Top 10 Products

In [5]:
q = '''
SELECT Product_Code,
       SUM(Order_Demand) AS total_demand
FROM demand
GROUP BY Product_Code
ORDER BY total_demand DESC
LIMIT 10
'''
pd.read_sql(q, conn)

,Product_Code,total_demand
0,Product_1359,472474000
1,Product_1248,289117000
2,Product_0083,210651000
3,Product_1341,169777000
4,Product_1295,123303000
5,Product_1241,117741000
6,Product_1245,103537000
7,Product_1286,101566400
8,Product_1432,97385000
9,Product_1274,92831000


## Query 5 – Monthly Demand Trend

In [6]:
q = '''
SELECT strftime('%Y-%m', Date) AS month,
       SUM(Order_Demand)       AS monthly_demand
FROM demand
WHERE Date IS NOT NULL
GROUP BY month
ORDER BY month
'''
pd.read_sql(q, conn).tail(24)

,month,monthly_demand
0,2011-01,429275083
1,2011-05,414698357
2,2011-06,433012741
3,2011-09,400046059
4,2011-10,451106723
5,2011-11,432423360
6,2011-12,429264786
7,2012-02,406596814
8,2012-03,466030426
9,2012-04,424941008


## Query 6 – Warehouse × Category Breakdown

In [7]:
q = '''
SELECT Warehouse,
       Product_Category,
       SUM(Order_Demand) AS total_demand
FROM demand
GROUP BY Warehouse, Product_Category
ORDER BY total_demand DESC
'''
pd.read_sql(q, conn).head(20)

,Warehouse,Product_Category,total_demand
0,Whse_J,Category_019,2739562587
1,Whse_S,Category_019,872342758
2,Whse_C,Category_019,521898473
3,Whse_J,Category_006,318766710
4,Whse_J,Category_005,128924470
5,Whse_A,Category_019,117403787
6,Whse_J,Category_007,112157242
7,Whse_S,Category_005,57097050
8,Whse_J,Category_033,42610000
9,Whse_S,Category_006,42059693


## Query 7 – Yearly Summary

In [8]:
q = '''
SELECT strftime('%Y', Date) AS year,
       SUM(Order_Demand)    AS yearly_demand,
       COUNT(*)             AS order_count
FROM demand
WHERE Date IS NOT NULL
GROUP BY year
ORDER BY year
'''
pd.read_sql(q, conn)

,year,yearly_demand,order_count
0,2011,8363894,640
1,2012,949259991,203635
2,2013,1014087922,218298
3,2014,1071178367,216404
4,2015,1099398391,209661
5,2016,991590399,188645
6,2017,294967,53


In [9]:
conn.close()
print('Done')

Done
